In [39]:
from my_api_info import youtube_api_key as apikey

In [2]:
!pip install google-api-python-client pandas

In [5]:
!pip install isodate

In [9]:
from googleapiclient.discovery import build
import pandas as pd

API_KEY = apikey

youtube = build("youtube", "v3", developerKey=API_KEY)

In [10]:

import pandas as pd

# Step 1: Search for fitness and nutrition Shorts

request = youtube.search().list(
    part="snippet",
    q="fitness OR nutrition",
    type="video",
    videoDuration="short",
    publishedAfter="2025-09-01T00:00:00Z",
    publishedBefore="2026-01-01T00:00:00Z",
    maxResults=50
)

response = request.execute()

print("Search results returned:", len(response["items"]))


# Step 2: Extract video IDs

video_ids = [
    item["id"]["videoId"]
    for item in response["items"]
]


# Step 3: Retrieve video information

video_response = youtube.videos().list(
    part="snippet,statistics,contentDetails",
    id=",".join(video_ids)
).execute()


# Step 4: Organize video information

videos = []

for item in video_response["items"]:

    videos.append({
        "video_id": item["id"],
        "title": item["snippet"]["title"],
        "channel": item["snippet"]["channelTitle"],
        "published_at": item["snippet"]["publishedAt"],
        "views": item["statistics"].get("viewCount"),
        "likes": item["statistics"].get("likeCount"),
        "comments": item["statistics"].get("commentCount"),
        "duration": item["contentDetails"]["duration"]
    })


df = pd.DataFrame(videos)


# Step 5: Filter for Shorts (60 seconds or less)

def duration_to_seconds(duration):
    import re

    hours = re.search(r"(\d+)H", duration)
    minutes = re.search(r"(\d+)M", duration)
    seconds = re.search(r"(\d+)S", duration)

    total = 0

    if hours:
        total += int(hours.group(1)) * 3600
    if minutes:
        total += int(minutes.group(1)) * 60
    if seconds:
        total += int(seconds.group(1))

    return total


df["duration_seconds"] = df["duration"].apply(duration_to_seconds)

df = df[df["duration_seconds"] <= 60]

# Step 6: Display results

print("\nQualifying Shorts:", len(df))

print(df.head(50))

# Step 7: Save results

df.to_csv("fitness_nutrition_shorts.csv", index=False)


Search results returned: 50

Qualifying Shorts: 41
       video_id                                              title  \
0   iJlO3WNLVZs  Complete vs Incomplete Proteins: What's the Di...   
1   UU1SZyexwlQ  #swimming #swim #fitness #weightloss #weightlo...   
2   Mdoz2uCxNLk  Strengthen Your Core 💪 #shorts #yoga #fitness ...   
3   quth7Bd5Vk8  Peak week shreds 💪 #peakweek #bodybuilding #pr...   
6   sSqrX-ApDX0  Full Body Sculpt 360 Workout #fitness #fullbod...   
7   3qd5U04h1WY  #gymmotivation #workout #fitness #athlete #bod...   
8   oTSTWuhy-oM  20-Minute Belly Burner #coreworkout #abworkout...   
10  09wcXpjYlmQ  How did we do on this transition? 🤩 #gymmotiva...   
11  1mlbdNzgO6A  What a crazy 4 months it has been! #shorts #gy...   
12  _Op0WgHe-5g  Small arms😱💔 #gym #gymmotivation #fitness #bic...   
13  70UxdosjsXo          Comeback to fitness 🔥 #lakshdeep #fitness   
14  EVyaHRdHDEk  The rhythm of bungee fitness! #workout #fitnes...   
17  A-rck-xrgkA  Full Body 400 #homewor

In [11]:

import pandas as pd

# Step 1: Retrieve information using your 50 video IDs

video_response = youtube.videos().list(
    part="snippet,statistics,contentDetails",
    id=",".join(video_ids)
).execute()


# Step 2: Organize the information

videos = []

for item in video_response["items"]:

    videos.append({
        "video_id": item["id"],
        "title": item["snippet"]["title"],
        "channel": item["snippet"]["channelTitle"],
        "views": item["statistics"].get("viewCount"),
        "likes": item["statistics"].get("likeCount"),
        "comments": item["statistics"].get("commentCount"),
        "duration": item["contentDetails"]["duration"]
    })


# Step 3: Create a DataFrame

df = pd.DataFrame(videos)


# Step 4: Display results

print(df.head(50))

print("\nVideos retrieved:", len(df))


# Step 5: Save your dataset

df.to_csv("youtube_fitness_nutrition_data.csv", index=False)


       video_id                                              title  \
0   iJlO3WNLVZs  Complete vs Incomplete Proteins: What's the Di...   
1   UU1SZyexwlQ  #swimming #swim #fitness #weightloss #weightlo...   
2   Mdoz2uCxNLk  Strengthen Your Core 💪 #shorts #yoga #fitness ...   
3   quth7Bd5Vk8  Peak week shreds 💪 #peakweek #bodybuilding #pr...   
4   AbY6l85SVKI  How to unf*ck your nutrition to get your dream...   
5   b_5yaiDti-4  Bang Bang Chicken Tenders… Here’s how I got on...   
6   sSqrX-ApDX0  Full Body Sculpt 360 Workout #fitness #fullbod...   
7   3qd5U04h1WY  #gymmotivation #workout #fitness #athlete #bod...   
8   oTSTWuhy-oM  20-Minute Belly Burner #coreworkout #abworkout...   
9   tHTT91WcCiQ  I ranked the TOP 10 fitness trackers for athle...   
10  09wcXpjYlmQ  How did we do on this transition? 🤩 #gymmotiva...   
11  1mlbdNzgO6A  What a crazy 4 months it has been! #shorts #gy...   
12  _Op0WgHe-5g  Small arms😱💔 #gym #gymmotivation #fitness #bic...   
13  70UxdosjsXo     

In [12]:
# Create DataFrame
df = pd.DataFrame(videos)

# Display as a table
display(df)

,video_id,title,channel,views,likes,comments,duration
0,iJlO3WNLVZs,Complete vs Incomplete Proteins: What's the Di...,Builtbyveno,19970948,264171,3359,PT31S
1,UU1SZyexwlQ,#swimming #swim #fitness #weightloss #weightlo...,Sal,1384040,14953,377,PT8S
2,Mdoz2uCxNLk,Strengthen Your Core 💪 #shorts #yoga #fitness ...,Shambhala Yoga,2944012,15311,97,PT7S
3,quth7Bd5Vk8,Peak week shreds 💪 #peakweek #bodybuilding #pr...,Sydney Freeman,31685,1335,45,PT11S
4,AbY6l85SVKI,How to unf*ck your nutrition to get your dream...,Madelaine Rascan,429391,24221,648,PT2M9S
5,b_5yaiDti-4,Bang Bang Chicken Tenders… Here’s how I got on...,The Fitness Chef,4944469,227012,2204,PT1M55S
6,sSqrX-ApDX0,Full Body Sculpt 360 Workout #fitness #fullbod...,Mackenzie Wells Fitness,90607,1965,16,PT25S
7,3qd5U04h1WY,#gymmotivation #workout #fitness #athlete #bod...,slicey,4245601,84268,1039,PT12S
8,oTSTWuhy-oM,20-Minute Belly Burner #coreworkout #abworkout...,Mackenzie Wells Fitness,38025,1063,4,PT26S
9,tHTT91WcCiQ,I ranked the TOP 10 fitness trackers for athle...,Justin Peters,658833,9546,118,PT1M52S


In [84]:
# Save video IDs to a CSV file
pd.DataFrame({"video_id": video_ids}).to_csv(
    "saved_video_ids.csv", index=False
)

In [53]:
from navigator_api_key import navigator_api as nav

In [55]:
!pip install youtube-transcript-api

In [56]:
from youtube_transcript_api import YouTubeTranscriptApi
import pandas as pd

ytt_api = YouTubeTranscriptApi()

In [79]:
video_ids = df["video_id"].tolist()

print(video_ids)

['iJlO3WNLVZs', 'UU1SZyexwlQ', 'Mdoz2uCxNLk', 'quth7Bd5Vk8', 'AbY6l85SVKI', 'b_5yaiDti-4', 'sSqrX-ApDX0', '3qd5U04h1WY', 'oTSTWuhy-oM', 'tHTT91WcCiQ', '09wcXpjYlmQ', '1mlbdNzgO6A', '_Op0WgHe-5g', '70UxdosjsXo', 'EVyaHRdHDEk', 'IAEKlSBAdlE', 'kWTjPyLKz04', 'A-rck-xrgkA', '63QJT4PZ6DY', 'QrmYXfJMQio', 'gwyPFxZhLnY', 'i7Bmc87byuU', '5uJSbJTXTck', 'K9smhcRTcfM', 'Hh5aX5ctV-g', 'HyKh4JeTtGg', 'eQIsPCyEHas', 'u2XocTiUiR0', 'LYvKEi-I_fs', 'g7ji1NCY0wY', '88dcLf2t0LI', 'w1YdTnDzUpI', 'e7NJP0icg8w', '0oLxZDJQ0OU', 'zwFaHAczeKs', 'QHNVT8sUY-M', 'hyQxWvRl-20', 'm-Sh9eCOQKw', 'xj3RHMGHsWc', 'ZHql1EIZifY', 'ydlYUo59zX8', '9AE5UPGMajw', '7ia_EddRMtk', 'TvATT48gHVE', 'Vfjh1TAmllU', '43_LYpfNXs0', 'xgezm9mB10I', '3O-tkNvEbS0', 'fLArOlSTjN0', 'bvD0Xe7ajys']


In [80]:
transcript_data = []

for video_id in video_ids:

    try:
        # Retrieve transcript
        transcript = ytt_api.fetch(video_id)

        # Combine transcript segments
        full_transcript = " ".join(
            segment.text for segment in transcript
        )

        # Store video ID and transcript
        transcript_data.append({
            "video_id": video_id,
            "transcript": full_transcript
        })

        print(f"Successfully retrieved: {video_id}")

    except Exception as e:
        print(f"Could not retrieve {video_id}: {e}")

Successfully retrieved: iJlO3WNLVZs
Could not retrieve UU1SZyexwlQ: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=UU1SZyexwlQ! This is most likely caused by:

Subtitles are disabled for this video

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide the information needed to replicate the error. Also make sure that there are no open issues which already describe your problem!
Could not retrieve Mdoz2uCxNLk: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=Mdoz2uCxNLk! This is most likely caused by:

Subtitles are disabled for this video

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com

In [81]:
transcripts_df = pd.DataFrame(transcript_data)

display(transcripts_df)

,video_id,transcript
0,iJlO3WNLVZs,This is 30 grams of protein. This is also 30 g...
1,AbY6l85SVKI,It is time for us to unfuck your nutrition bec...
2,b_5yaiDti-4,You're really about to tell me that dieting is...
3,oTSTWuhy-oM,Look at the world. [Music] Bring it up. Bring ...
4,tHTT91WcCiQ,If you put a gun to my head and had me rank th...
5,09wcXpjYlmQ,[music] Desperado sitting [music] in a old Mon...
6,1mlbdNzgO6A,[Music] I'm going to show you how great I am. ...
7,_Op0WgHe-5g,Go. Go. Go. Go. Go. >> [music]
8,IAEKlSBAdlE,All the girls here come both ways to sound. >>...
9,kWTjPyLKz04,I ask people who are really fit what they do f...


In [82]:
print("Total transcripts retrieved:", len(transcripts_df))

transcripts_df["word_count"] = (
    transcripts_df["transcript"].str.split().str.len()
)

display(transcripts_df[["video_id", "word_count"]])

Total transcripts retrieved: 22


,video_id,word_count
0,iJlO3WNLVZs,66
1,AbY6l85SVKI,443
2,b_5yaiDti-4,352
3,oTSTWuhy-oM,16
4,tHTT91WcCiQ,415
5,09wcXpjYlmQ,9
6,1mlbdNzgO6A,113
7,_Op0WgHe-5g,7
8,IAEKlSBAdlE,203
9,kWTjPyLKz04,268


In [65]:
models = client.models.list()

for model in models.data:
    print(model.id)

flux.1-dev
flux.1-schnell
flux.2-klein
gemma-4-31b-it
gpt-oss-120b
gpt-oss-20b
granite-3.3-8b-instruct
kokoro
llama-3.3-70b-instruct
medgemma-27b-it
meta-muse-glimmer-30b
mistral-small-3.1
nemotron-3-nano-30b-a3b
nemotron-3-super-120b-a12b
nomic-embed-text-v1.5
sfr-embedding-mistral
whisper-large-v3


In [66]:
MODEL = "gemma-4-31b-it"

In [67]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Count from 1 to 5."}
    ],
    temperature=0
)

print(response.choices[0].message.content)

1, 2, 3, 4, 5


In [75]:
import json

def analyze_transcript(transcript):

    prompt = f"""
You are conducting a research study on YouTube video
communication characteristics.

Analyze the following transcript and return counts for
three variables.

VARIABLE 1: viewer_questions
Count the number of questions directed toward viewers.

Rules:
- Include questions intended to engage the audience.
- Include questions asking viewers about their opinions,
  experiences, or preferences.
- EXCLUDE rhetorical questions.
- EXCLUDE questions the creator asks themselves.
- EXCLUDE questions that are purely informational or
  used to introduce a topic without audience engagement.

VARIABLE 2: calls_to_action
Count explicit requests asking viewers to:
- Like the video
- Comment
- Subscribe
- Share the video

Rules:
- Count each distinct requested action.
- If the creator asks viewers to like AND subscribe,
  count that as two calls to action.
- Count repeated requests if they occur at separate
  points in the video.
- Do not count casual mentions of these actions unless
  the creator explicitly requests them.

VARIABLE 3: personal_storytelling
Count distinct references to the creator's own personal
experiences.

Rules:
- Include personal anecdotes and stories.
- Include references to the creator's own past events,
  experiences, or activities.
- Exclude hypothetical examples.
- Exclude stories about other people unless the creator
  is describing their own involvement.
- Do not count every sentence in one story separately.
  Count one distinct personal experience as one instance.

IMPORTANT:
- Analyze ONLY the transcript provided.
- Do not invent or assume information.
- Do not estimate counts.
- Avoid double-counting.
- If a category has no occurrences, return 0.
- Include brief evidence excerpts supporting each count.

Return ONLY valid JSON in this exact structure:

{{
    "viewer_questions": 0,
    "calls_to_action": 0,
    "personal_storytelling": 0,
    "viewer_question_evidence": [],
    "call_to_action_evidence": [],
    "personal_storytelling_evidence": []
}}

TRANSCRIPT:
{transcript}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a careful research assistant performing quantitative content analysis. Follow the coding rules exactly."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_tokens=1000
    )

    result = response.choices[0].message.content

    # Remove Markdown code fences
    result = result.strip()

    if result.startswith("```json"):
        result = result[7:]

    elif result.startswith("```"):
        result = result[3:]

    if result.endswith("```"):
        result = result[:-3]

    # Convert JSON into Python dictionary
    return json.loads(result.strip())

In [76]:
first_transcript = transcripts_df.iloc[0]["transcript"]

test_result = analyze_transcript(first_transcript)

print(json.dumps(test_result, indent=4))

{
    "viewer_questions": 0,
    "calls_to_action": 0,
    "personal_storytelling": 0,
    "viewer_question_evidence": [],
    "call_to_action_evidence": [],
    "personal_storytelling_evidence": []
}


In [77]:
first_transcript = transcripts_df.iloc[1]["transcript"]

test_result = analyze_transcript(first_transcript)

print(json.dumps(test_result, indent=4))

{
    "viewer_questions": 0,
    "calls_to_action": 0,
    "personal_storytelling": 1,
    "viewer_question_evidence": [],
    "call_to_action_evidence": [],
    "personal_storytelling_evidence": [
        "If you want to lose fat, but you like eating big ass portions of food like me"
    ]
}


In [83]:
# Create an empty list to store the results
llm_results = []

# Loop through every video and its transcript
for index, video in transcripts_df.iterrows():
    try:
        # Analyze the transcript
        analysis = analyze_transcript(video["transcript"])

        # Add the video ID to the results
        analysis["video_id"] = video["video_id"]

        # Store the results
        llm_results.append(analysis)

        print(f"Successfully analyzed video {video['video_id']}")

    except Exception as e:
        print(f"Error analyzing video {video['video_id']}: {e}")

# Convert the results into a DataFrame
llm_df = pd.DataFrame(llm_results)

# Display the results as a table
display(llm_df)

Successfully analyzed video iJlO3WNLVZs
Successfully analyzed video AbY6l85SVKI
Successfully analyzed video b_5yaiDti-4
Successfully analyzed video oTSTWuhy-oM
Successfully analyzed video tHTT91WcCiQ
Successfully analyzed video 09wcXpjYlmQ
Successfully analyzed video 1mlbdNzgO6A
Successfully analyzed video _Op0WgHe-5g
Successfully analyzed video IAEKlSBAdlE
Successfully analyzed video kWTjPyLKz04
Successfully analyzed video A-rck-xrgkA
Successfully analyzed video 5uJSbJTXTck
Successfully analyzed video Hh5aX5ctV-g
Successfully analyzed video 88dcLf2t0LI
Successfully analyzed video w1YdTnDzUpI
Successfully analyzed video e7NJP0icg8w
Successfully analyzed video 0oLxZDJQ0OU
Successfully analyzed video zwFaHAczeKs
Successfully analyzed video hyQxWvRl-20
Successfully analyzed video xj3RHMGHsWc
Successfully analyzed video 7ia_EddRMtk
Successfully analyzed video fLArOlSTjN0


,viewer_questions,calls_to_action,personal_storytelling,viewer_question_evidence,call_to_action_evidence,personal_storytelling_evidence,video_id
0,0,0,0,[],[],[],iJlO3WNLVZs
1,0,0,1,[],[],"[If you want to lose fat, but you like eating ...",AbY6l85SVKI
2,1,0,0,[Did you get all that?],[],[],b_5yaiDti-4
3,0,0,0,[],[],[],oTSTWuhy-oM
4,0,0,3,[],[],"[I tested this one [Fitbit Charge 6]., I used ...",tHTT91WcCiQ
5,0,0,0,[],[],[],09wcXpjYlmQ
6,0,0,0,[],[],[],1mlbdNzgO6A
7,0,0,0,[],[],[],_Op0WgHe-5g
8,0,0,2,[],[],[Mine says that my resting calories are betwee...,IAEKlSBAdlE
9,0,0,0,[],[],[],kWTjPyLKz04
